# Capstone — refresh-priority research paper

This executable paper mirror backs the deployed public page. It contains aggregate, public-safe evidence only.

## 1. Question

Given limited review capacity, which content items should a reviewer inspect first for a possible refresh? The unit is one anonymized content item. The queue is decision support for a human reviewer; Precision@50 is primary because the reviewer has a finite shortlist.

## 2. Data

The bundled FlyRank ML Internship starter release contains 30,000 pseudonymized content items from 32 pseudonymized clients, with trailing-90-day aggregates and embedded comparison windows. This notebook uses aggregate model artifacts only. Client/content identifiers are grouping-only; titles, domains, URLs, keywords, raw queries, provider/model fields, product flags, outcome-window columns, `trend_direction`, and `trend_pct` are excluded from features or publication.

In [ ]:
import json
from pathlib import Path

results = json.loads(Path('outputs/model_results.json').read_text())
assert results['input_rows'] == 30000
assert results['split_strategy'] == 'client_holdout'
print(f"Rows: {results['input_rows']:,}; client-held-out test rows: {results['test_rows']:,}")
print(f"Proxy-label base rate: {results['target_positive_rate']:.3f}")

## 3. Methodology

The proxy target is `is_declining_label = 1` when the release-defined `trend_direction` is down (more than 20% lower impressions across its comparison windows). It is contemporaneous and rule-defined, not a future outcome. The reproducible pipeline uses 18 observable numeric and eight categorical/bucketed features (52 after encoding), seed 42, and a whole-client holdout. The transparent baseline weights visibility 40%, freshness risk 30%, position opportunity 25%, and depth gap 5%. Deliberate leak tests show that current-window outcome fields create suspiciously perfect scores, so those fields remain excluded.

## 4. Results (vs baseline)

Every number below is computed on the same held-out clients. Higher Precision@50 is better; the test proxy-label base rate is printed alongside the comparison.

In [ ]:
baseline = results['baseline']
for name, metric in results['models'].items():
    print(f"{name:20} AUC={metric['roc_auc']:.3f}  AP={metric['average_precision']:.3f}  P@50={metric['precision_at_50']:.3f}")
print(f"{'baseline_rules':20} AUC={baseline['baseline_roc_auc']:.3f}  AP={baseline['baseline_average_precision']:.3f}  P@50={baseline['baseline_precision_at_50']:.3f}")
assert results['models']['random_forest']['precision_at_50'] > baseline['baseline_precision_at_50']

## 5. Limitations

This result ranks similarity to a proxy label; it does not show that a refresh causes recovery or that the model predicts a search engine. One split and seed do not measure uncertainty. Feature importance is associational, not causal. Results from the starter slice are not claimed as full-warehouse or production performance. Human review retains responsibility for context, quality, compliance, and seasonality.

## 6. Ranked recommendations

1. Human-review the top 50 random-forest candidates first.
2. Use reason codes to route refresh, CTR, engagement, or monitoring checks—never as automatic edits.
3. Keep the rule baseline as an inspectable control.
4. Before rollout, run repeated client-held-out and time-forward validation with a future outcome.
5. Monitor base rate, Precision@50, and drift; retire the queue if it ceases to beat its control.

## 7. Artifacts the paper embeds

The deployed page presents the aggregate model comparison and top feature importances in accessible HTML charts with text alternatives. The generated SVG artifacts remain under `outputs/charts/`.

## Reproducibility

From a fresh clone: `pip install -r requirements.txt`, then `python scripts/run_all.py`. The model seed is 42.

## Acknowledgments & data credit

Built on the [FlyRank ML Internship dataset](https://flyrank.ai).

## Week-8 showcase demo outline (5 minutes)

**0:00–0:45 — Question and case study.** FlyRank content reviewers cannot inspect every existing item in a cycle. My question was: given roughly 50 review slots, which items should be inspected first for a potential refresh?

**0:45–1:45 — Method.** I treated this as a ranking problem using an anonymized 30,000-item FlyRank starter release. I compared a transparent weighted rule with logistic regression, a decision tree, and a random forest, using a fixed seed and a client-held-out test split so whole pseudonymized clients were unseen during evaluation.

**1:45–3:00 — One chart.** Show the held-out Precision@50 chart: rule baseline 0.24, logistic regression 0.40, decision tree 0.62, and random forest 0.74; show the 0.54 proxy-label base rate beside it. Explain that Precision@50 matches the size of the reviewer’s shortlist.

**3:00–4:00 — One honest result.** On this one client-held-out split, the random forest placed 37 proxy-declining items in its first 50 recommendations, versus 12 for the rule baseline. This measures ranking quality against a contemporaneous proxy label, not whether a refresh causes a recovery.

**4:00–5:00 — Recommendation.** Use the top 50 as a human-review queue with reason codes, keep the transparent baseline as a control, and validate on repeated time-forward splits before production use. Close with the live paper link.

## Shareable cuts

### Social post

I turned a FlyRank content-review bottleneck into a ranking problem: when a reviewer has about 50 slots, which existing items should they inspect first? On an anonymized 30,000-item starter release, I compared a transparent rule with three models on a client-held-out split and evaluated them with Precision@50—the metric that actually matches the shortlist. The random forest measured 0.74 Precision@50 versus 0.24 for the rule baseline, but I frame it as a human-review aid against a proxy decline label, not proof that refreshing content causes recovery. Read the case study: https://justqwertty.github.io/flyrankAssignment1/

### Employer-facing summary

I built a public-safe content-refresh prioritization system that ranks which existing items a FlyRank reviewer should inspect first, with reason codes and a transparent rule baseline. I used an anonymized 30,000-item FlyRank ML Internship starter release and evaluated logistic regression, decision tree, and random-forest models on a client-held-out split. The selected random forest measured Precision@50 of 0.74 versus 0.24 for the baseline, which is decision-support evidence for triaging human review against a proxy decline label—not a causal claim about refresh outcomes.